# Qwen3-ASR Audio Playground

In [ ]:
!pip install -U -q "transformers>=5.13.0" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec

## Setup

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

## Kaldi Test Data

In [ ]:
import torchaudio
from IPython.display import Audio, display

dataset = []

dataset_path = "/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test" #other languages *_de *_fr *_es

# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/eb/test *_de *_fr *_es
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/de/test *_de *_fr *_es 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test *_de *_fr *_es

wav_scp_dict = {}

with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}

with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

first_entry = dataset[0]

print(first_entry)
print("-"*10)
print(first_entry["text"])

audio = torchaudio.load(first_entry["audio"])[0]

display(Audio(audio, rate=16000))

## Transcription Function (Context Changeable)

In [15]:
def transcribe(audio_path, prompt=None, language=None):
    kwargs = {"audio": audio_path}
    if prompt:
        kwargs["prompt"] = prompt
    if language:
        kwargs["language"] = language
    inputs = processor.apply_transcription_request(**kwargs).to(model.device, model.dtype)

    output_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    return processor.decode(generated_ids, return_format="transcription_only")[0].strip()

## Test on a Single Audio from the Test Dataset

In [ ]:
sample = dataset[0]

model_output = transcribe(sample["audio"])

print("SAMPLE :", sample)
print("LABEL :", sample["text"])
print("MODEL_OUTPUT :", model_output)

## Adding Context

In [ ]:
sample_path = "/training-1/asr/voxpopuli/out/transcribed_data/en/2018/20180704-0900-PLENARY-en_20180704-17:51:01_12.wav"
sample_text = "without these mistakes that you made we would not be debating this issue here today so what im asking mr kurz to do is to upscale this strategy to an eur one hundred and twenty million anti poverty strategy for all european citizens"

wav, sr = torchaudio.load(sample_path, normalize=True)
display(Audio(wav, rate=sr))

prompt = "Names: Mr. Kurz"   # e.g. "mr kurz, european citizens"

print(transcribe(sample_path))
print("-"*10)
print(transcribe(sample_path, prompt=prompt))